In [13]:
import pandas as pd
from scipy.io import arff
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

# === Load and preprocess dataset ===
df = pd.read_csv("compas_synthetic_data_1000_200_epochs.csv") 

# Convert byte string columns
for col in df.columns:
    if df[col].dtype == object:
        df[col] = df[col].apply(lambda x: x.decode("utf-8") if isinstance(x, bytes) else x)

# Convert categorical/binary features to int
categorical_columns = ["sex",
    "age_cat_25-45", "age_cat_Greaterthan45", "age_cat_Lessthan25",
    "race_African-American", "race_Caucasian",
    "c_charge_degree_F", "c_charge_degree_M"
]
for col in categorical_columns:
    df[col] = df[col].astype(int)

# === Split data ===
X = df.drop(columns=["two_year_recid"])
y = df["two_year_recid"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# === Models and Evaluation ===
results = {}

# 1. Decision Tree
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)
results["Decision Tree"] = accuracy_score(y_test, dt.predict(X_test))

# 2. Logistic Regression
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)
results["Logistic Regression"] = accuracy_score(y_test, lr.predict(X_test))

# 3. Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
results["Random Forest"] = accuracy_score(y_test, rf.predict(X_test))

# 4. SVM
svm = SVC()
svm.fit(X_train, y_train)
results["SVM"] = accuracy_score(y_test, svm.predict(X_test))

# 5. XGBoost
xgb = XGBClassifier(eval_metric="logloss", random_state=42)
xgb.fit(X_train, y_train)
results["XGBoost"] = accuracy_score(y_test, xgb.predict(X_test))

# === Display Results ===
print("\nAccuracy of models (TS-TS) Our Prompt With Fairness:")
for model, acc in results.items():
    print(f"{model}: {acc:.4f}")

# === Add predictions to test set ===
X_test = X_test.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)
X_test_copy = X_test.copy()
X_test_copy["two_year_recid"] = y_test

# Add predictions
X_test_copy["pred_decision_tree"] = dt.predict(X_test)
X_test_copy["pred_logistic_regression"] = lr.predict(X_test)
X_test_copy["pred_random_forest"] = rf.predict(X_test)
X_test_copy["pred_svm"] = svm.predict(X_test)
X_test_copy["pred_xgboost"] = xgb.predict(X_test)

# === Save to CSV ===
X_test_copy.to_csv("Synt_Data_DECAF_ with_predictions.csv", index=False)
print("\nPredictions have been added and saved.")


Accuracy of models (TS-TS) Our Prompt With Fairness:
Decision Tree: 0.5400
Logistic Regression: 0.5950
Random Forest: 0.5350
SVM: 0.5400
XGBoost: 0.6000

Predictions have been added and saved.


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

# === Load full real dataset
df = pd.read_csv("Real_MIMIC.csv", low_memory=False)

# === Drop rows with missing los_seconds and binarize label
df = df.dropna(subset=["los_seconds"])
df["los_seconds"] = pd.to_numeric(df["los_seconds"], errors="coerce")
df = df.dropna(subset=["los_seconds"])
df["label"] = (df["los_seconds"] >= 345600).astype(int)

# === Drop columns with too many missing values (>90%)
df = df.loc[:, df.isnull().mean() < 0.9]

# === Sample 10% of real data with label distribution preserved
df_sampled = df.groupby("label", group_keys=False).apply(lambda x: x.sample(frac=1, random_state=42)).reset_index(drop=True)

# === Separate features and labels
X = df_sampled.drop(columns=["los_seconds", "label"])
y = df_sampled["label"]

# === Encode categorical features (object/bool) and impute missing values
for col in X.select_dtypes(include=["object", "bool"]).columns:
    unique_vals = X[col].dropna().unique().tolist()
    if set(unique_vals).issubset({'True', 'False', 'true', 'false'}):
        X[col] = X[col].astype(str).map({'True': 1, 'False': 0, 'true': 1, 'false': 0})
    else:
        X[col] = LabelEncoder().fit_transform(X[col].astype(str))

# === Impute remaining NaNs (mean for numeric)
imputer = SimpleImputer(strategy='mean')
X = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)

# === Train-test split (80-20)
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

# === Define models
models = {
    'pred_decision_tree': DecisionTreeClassifier(random_state=42),
    'pred_logistic_regression': LogisticRegression(max_iter=1000, random_state=42),
    'pred_random_forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'pred_svm': SVC(probability=True, random_state=42),
    'pred_xgboost': XGBClassifier(eval_metric='logloss', random_state=42)
}

# === Create a test set copy to hold predictions
df_test_with_preds = X_test.copy()
df_test_with_preds["true_label"] = y_test.values

# === Train and add predictions
for name, model in models.items():
    model.fit(X_train, y_train)
    df_test_with_preds[name] = model.predict(X_test)

# === Save results
df_test_with_preds.to_csv("real_test_with_predictions.csv", index=False)
print("✅ Saved predictions to 'real_test_with_predictions.csv'")


C:\Users\SSHAKI~1\AppData\Local\Temp/ipykernel_13224/1100983036.py:25: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_sampled = df.groupby("label", group_keys=False).apply(lambda x: x.sample(frac=1, random_state=42)).reset_index(drop=True)


: 